[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-1/lab-1.4-pipelining-profile.ipynb)

# LAB·1.4 · Pipelining and the profile

**Hardware:** TPU runtime for everything measured; the reasoning transfers anywhere.

You never wrote a DMA in labs 1.1-1.3, yet the compiled kernels overlap loads with compute. TPU grid steps are a software pipeline: while step k computes on its blocks, step k+1's blocks stream in. This lab makes that visible in a profile instead of taking it on faith. The site's EX·02 instrument animates exactly this schedule.

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
INTERP = not ON_TPU  # interpret mode anywhere; compiled kernels on a real TPU

def check(name, got, want, tol=2e-2):
    err = float(jnp.abs(got.astype(jnp.float32) - want.astype(jnp.float32)).max())
    status = "ok" if err <= tol else "FAIL"
    print(f"{name}: max err {err:.3e} [{status}]")
    assert err <= tol, name


## See the pipeline in a trace

`jax.profiler.trace` writes a TensorBoard-loadable trace. On Colab: run the cell, then Runtime → open TensorBoard on the logdir, find the matmul, and look at the DMA rows against the MXU row. The question to answer from the picture: what fraction of the step is the MXU actually busy?

In [ ]:
# the LAB 1.2 kernel, inlined so this notebook stands alone
def matmul_kernel(a_ref, b_ref, o_ref):
    k = pl.program_id(2)
    @pl.when(k == 0)
    def _init():
        o_ref[...] = jnp.zeros_like(o_ref)
    o_ref[...] += jnp.dot(a_ref[...], b_ref[...], preferred_element_type=jnp.float32).astype(o_ref.dtype)

def matmul(a, b, bm=128, bn=128, bk=128):
    m, k = a.shape
    _, n = b.shape
    return pl.pallas_call(
        matmul_kernel,
        grid=(m // bm, n // bn, k // bk),
        in_specs=[pl.BlockSpec((bm, bk), lambda i, j, kk: (i, kk)),
                  pl.BlockSpec((bk, bn), lambda i, j, kk: (kk, j))],
        out_specs=pl.BlockSpec((bm, bn), lambda i, j, kk: (i, j)),
        out_shape=jax.ShapeDtypeStruct((m, n), a.dtype),
        interpret=INTERP,
    )(a, b)

if ON_TPU:
    N = 4096
    a = jax.random.normal(jax.random.key(0), (N, N), jnp.bfloat16)
    b = jax.random.normal(jax.random.key(1), (N, N), jnp.bfloat16)
    mm = jax.jit(lambda x, y: matmul(x, y, 512, 512, 512))
    mm(a, b).block_until_ready()
    with jax.profiler.trace("/tmp/pallas-trace"):
        mm(a, b).block_until_ready()
    print("trace written to /tmp/pallas-trace; open TensorBoard on it")
else:
    print("Profiling needs a TPU runtime.")

## Starve the pipeline on purpose

Shrink blocks until DMA setup dominates and the MXU idles; grow them until VMEM overflows and the compiler refuses. Record the failure point: from stage 0's constants, predict the VMEM budget line `3 blocks in flight × block bytes ≤ VMEM` and check your prediction against where the compile actually fails.

## The habit

From here on, every kernel you write gets one profile look before you call it done. A kernel that is correct but 40% MXU-idle is not done; the profile says why.

## Read what Mosaic sees

One layer below `pallas_call` your kernel is a Mosaic module: MLIR in the `tpu` dialect, the last layer of this stack you can read (the dialect and much of its lowering are open source in the jax repo under `jaxlib/mosaic`; LLO below it is not). `debug=True` makes Pallas print the kernel jaxpr and the module at lowering time.

Three things to find in the output: your BlockSpec index maps compiled into `@transform` functions, your `pl.when` compiled into a real `scf.if`, and your `jnp.dot` as `tpu.matmul` with the dimension numbers intact. This is also the vocabulary the museum's lattice and VMEM errors speak.


In [ ]:
# the LAB 1.2 kernel again, with debug=True: lowering prints the kernel
# jaxpr and the Mosaic module before any machine code exists
def matmul_debug(a, b, bm=256, bn=256, bk=256):
    m, k = a.shape
    _, n = b.shape
    return pl.pallas_call(
        matmul_kernel,
        grid=(m // bm, n // bn, k // bk),
        in_specs=[pl.BlockSpec((bm, bk), lambda i, j, kk: (i, kk)),
                  pl.BlockSpec((bk, bn), lambda i, j, kk: (kk, j))],
        out_specs=pl.BlockSpec((bm, bn), lambda i, j, kk: (i, j)),
        out_shape=jax.ShapeDtypeStruct((m, n), a.dtype),
        debug=True,
    )(a, b)

x = jax.ShapeDtypeStruct((512, 512), jnp.bfloat16)
if jax.devices()[0].platform == "tpu":
    jax.jit(matmul_debug).lower(x, x)  # lowering alone triggers the debug print
else:
    # no TPU attached: cross-lower for the tpu platform; the Mosaic
    # passes ship in jaxlib, so this works on any machine
    import jax.export
    jax.export.export(jax.jit(matmul_debug), platforms=["tpu"])(x, x)
